## Part 5: Inference function

`predict_podium(record)` answers one question: will this driver finish on the podium (top 3) in this race? It takes the pre-race features of **one driver in one race** and returns the podium probability and a yes/no answer. It uses the final model from Part 4, so nothing is retrained.

In [ ]:
# LOCAL SETUP (dropped when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import POST_RACE_COLS
from IPython.utils.capture import capture_output
with capture_output():
    %run p4_models.ipynb

### 5.1 The function

* **Input:** a dictionary with the features in `FEATURES` (Part 3). A feature that is unknown can be left out or set to `None`.
* **Step 1:** refuse the record if it contains a post-race column (`positionOrder`, `points`, `status`, ...), because those are not known before the race.
* **Step 2:** build one row with the features in the same order as in training.
* **Step 3:** give the row to `FINAL_MODEL`. The model already contains the preprocessing from Part 4 (filling missing values, scaling), so it is not repeated here.
* **Output:** the podium probability, the yes/no answer (probability at least `FINAL_THRESHOLD`) and the list of missing features.

In [ ]:
import numpy as np
import pandas as pd

def predict_podium(record, model=None, threshold=None):
    model = FINAL_MODEL if model is None else model
    threshold = FINAL_THRESHOLD if threshold is None else threshold

    leaky = set(record.keys()) & (set(POST_RACE_COLS) | {"podium"})
    if leaky:
        raise ValueError(f"post-race information is not allowed: {sorted(leaky)}")

    values = {}
    for f in FEATURES:
        v = record.get(f, np.nan)
        values[f] = np.nan if pd.isna(v) else v
    X = pd.DataFrame([values], columns=FEATURES).astype(float)

    proba = float(model.predict_proba(X)[0, 1])
    return {
        "podium_probability": proba,
        "podium_prediction": int(proba >= threshold),
        "missing_features": [f for f in FEATURES if pd.isna(values[f])],
    }

### 5.2 Example A: complete record
A real driver-race from the validation years: the first one where all features are known. The finishing position is only printed to describe the example, it is not an input.

In [ ]:
def describe_row(row):
    driver = tables["drivers"].set_index("driverId").loc[row["driverId"], "surname"]
    race = tables["races"].set_index("raceId").loc[row["raceId"], "name"]
    finish = outcomes.set_index(["raceId", "driverId"]).loc[(row["raceId"], row["driverId"]), "positionOrder"]
    return f"{driver}, {race} {row['year']} (actual finish: P{int(finish)})"

complete_rows = table[(table["split"] == "validation") & table[FEATURES].notna().all(axis=1)]
row_A = complete_rows.iloc[0]
record_A = row_A[FEATURES].to_dict()

print(describe_row(row_A))
print("input:", record_A)
print("output:", predict_podium(record_A))

All features are given, so the model uses them as they are. The output has the podium probability and the yes/no answer at the validation threshold. The function running without errors does not show that the prediction is right: one record proves nothing about accuracy. The accuracy of the model is measured in Part 4 on the validation and test sets.

### 5.3 Example B: partially missing record
A real driver-race where information is missing: the first validation row of a driver's first Grand Prix. He has no earlier races (no form) and it is the first round of the season (no standings yet). The unknown fields are left out of the dictionary.

In [ ]:
first_race_rows = table[(table["split"] == "validation") & table["driver_form5"].isna()]
row_B = first_race_rows.iloc[0]
record_B = {k: v for k, v in row_B[FEATURES].to_dict().items() if pd.notna(v)}

print(describe_row(row_B))
print("input:", record_B)
print("output:", predict_podium(record_B))

The missing features are listed in `missing_features`. The preprocessing inside the model fills each missing number with the median of the training years, and for features that had gaps in training it also adds a "was missing" flag, so the model knows the value was filled in. The prediction is based on a typical value, not the real one, so it is less reliable than for a complete record.

### 5.4 Checks
The function should give the same probability as calling the model directly, including for the validation rows that have missing values, and it should refuse post-race columns.

In [ ]:
val = table[table["split"] == "validation"]
direct = FINAL_MODEL.predict_proba(val[FEATURES])[:, 1]
one_by_one = np.array([predict_podium(r[FEATURES])["podium_probability"] for _, r in val.iterrows()])
print("validation rows:", len(val), "| largest difference:", abs(direct - one_by_one).max())
assert np.allclose(direct, one_by_one)

for col in ["positionOrder", "points", "status", "podium"]:
    try:
        predict_podium({**record_A, col: 1})
    except ValueError:
        print("refused:", col)

### 5.5 Limitations
* The function takes the features computed in Part 3 (form, standings, circuit history). It does not build them from raw race data.
* Missing values are replaced by typical values. Predictions for records with missing qualifying or grid information are not reliable.
* It predicts one driver at a time, so it can say "podium" for more or fewer than three drivers in the same race. It cannot be better than the Part 4 model.